# Tenrai probe

Issue #27. Jikan's public API was discontinued on 1 Oct 2026, so the pipeline will read MAL data from Tenrai (`https://api.tenrai.org/v1`, ADR 0011). Before designing the ingestion pipeline, this notebook measures what Tenrai returns and what it costs per anime.

1. **First look**: one request per endpoint to see the response shape.
2. **Probe pipeline**: fetches details, statistics, reviews (page 1) and all episode pages for each anime, one request at a time, 0.6 s apart (100 requests/min), logging every request.
3. **Runs**: 4 spread-out list pages (150 anime each) plus the 100 most-followed anime. 2,852 requests in total.
4. **Analysis**: reads the logs and checks rate compliance, latency, size, cost per anime, and what the data contains.

The findings are written up in `docs/METRICS.md`. The raw logs (`probe_*.json*`) are git-ignored.

## 1. First look at the API

One list request, then one `/full` request, to see the field names and the pagination object (total, `per_page`, last page).

In [ ]:
import httpx

client = httpx.Client(
 base_url="https://api.tenrai.org/v1/",
 timeout=30,
 headers={"User-Agent": "AnimePlanet https://github.com/Satveer27/Anime_App_Backend"},
)

r = client.get("anime")
print(r.json())

In [ ]:
# analysis of the response

print(r.status_code)
print(r.headers)
print(r.json().keys())
for key in r.json().get("data")[0].keys():
    print(key)

In [ ]:
for key in r.json().get("data")[0].keys():
    print(key)
    print(r.json().get("data")[0].get(key))


In [19]:
anime_id = r.json().get("data")[0].get("mal_id")
anime_full = client.get(f"anime/{anime_id}/full")

In [ ]:
for key in anime_full.json().get("data").keys():
    print(key)
    print(anime_full.json().get("data").get(key))

In [ ]:
print(r.json().get("pagination"))

## 2. Probe pipeline

`get_data` makes one request with retries (429 waits for `Retry-After`, 5xx and network errors back off) and logs it: start time, endpoint, status, latency, bytes, cache and rate-limit headers. `check_time_since_last_request` keeps request starts at least 0.6 s apart. The pipeline reads one or more list pages, and for each anime fetches `/full`, `/statistics`, `/reviews` and every `/episodes` page, saving results to `probe_results_*.jsonl` and the request log to `probe_metrics_*.json`.

In [33]:
import time
import json
import httpx
REQUEST_PER_MIN = 100
MIN_INTERVAL = 60 / REQUEST_PER_MIN

def endpoint_type(endpoint: str) -> str:
    path = endpoint.split("?")[0].strip("/")   # drop the query string
    parts = path.split("/")
    if parts == ["anime"]:
        return "list"
    return parts[-1]


def get_data(endpoint, metrics, client, max_retries=5, anime_id=None):
    if max_retries <= 0:
        print("Max retries exceeded. Exiting...")
        raise Exception("Max retries exceeded")

    try:
        start = time.perf_counter()
        last_request_time = time.monotonic()
        r = client.get(endpoint)
        latency = time.perf_counter() - start

        metrics.append({
                "id": anime_id,
                "endpoint": endpoint_type(endpoint),
                "status": r.status_code,
                "error" : None,
                "latency_s": latency,
                "bytes": len(r.content),
                "retry_after": r.headers.get("Retry-After"),
                "nginx_cache": r.headers.get("x-nginx-cache"),
                "ratelimit": {k: v for k, v in r.headers.items() if "ratelimit" in k.lower()},
                "started": last_request_time,
        })

        if r.status_code == 429:  # Rate limit exceeded
            retry_after = r.headers.get("Retry-After")
            wait = float(retry_after) if retry_after else 30
            print(f"Rate limit exceeded. Retrying after {wait} seconds...")
            time.sleep(wait)
            return get_data(endpoint, metrics, client, max_retries=max_retries - 1, anime_id=anime_id)

        keys = {500, 502, 503, 504}
        if r.status_code in keys:
            print(f"Server error {r.status_code}. Retrying...")
            time.sleep(2 ** (5 - max_retries))
            return get_data(endpoint, metrics, client, max_retries=max_retries - 1, anime_id=anime_id)

    except httpx.TransportError as e:
        metrics.append({
            "id": anime_id,
            "endpoint": endpoint_type(endpoint),
            "status": None,
            "error": type(e).__name__,  # e.g. "ReadTimeout", "ConnectError"
            "latency_s": time.perf_counter() - start,
            "bytes": 0,
            "retry_after": None,
            "nginx_cache": None,
            "ratelimit": {},
            "started": last_request_time,
        })
        print(f"Network error ({type(e).__name__}). Retrying...")
        time.sleep(2 ** (5 - max_retries))
        return get_data(endpoint, metrics, client, max_retries=max_retries - 1, anime_id=anime_id)
        

    return r, last_request_time


def check_time_since_last_request(last_request_time):
    elapsed = time.monotonic() - last_request_time
    if elapsed < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - elapsed)
    return time.monotonic()

def prone_pipeline(curr_limit=40, page=150, base_endpoint="anime?rating=g%2Cpg%2Cpg13%2Cr17%2Cr&limit=100", file_name = None):
    result = []
    anime_data_not_taken = []
    list_anime_data_not_taken = []
    metrics = []
    total_pages = None
    isDone = False
    last_request_time = time.monotonic()
    client = None

    if file_name:
        RESULTS_FILE = f"probe_results_{file_name}.jsonl"
        METRICS_FILE = f"probe_metrics_{file_name}.json"
    else:
        RESULTS_FILE = f"probe_results_page{page}.jsonl"
        METRICS_FILE = f"probe_metrics_page{page}.json"

    try:
        open(RESULTS_FILE, "w").close()
        open(METRICS_FILE, "w").close()


        client = httpx.Client(
            base_url="https://api.tenrai.org/v1/",
            timeout=30,
            headers={"User-Agent": "AnimePlanet https://github.com/Satveer27/Anime_App_Backend"},
        )

        while not isDone:
            try:
                sep = "&" if "?" in base_endpoint else "?"
                page_endpoint = f"{base_endpoint}{sep}page={page}"
                

                last_request_time = check_time_since_last_request(last_request_time)
                all_anime, last_request_time = get_data(page_endpoint, metrics, client)
                
                if all_anime.status_code != 200:
                    raise Exception(f"Error fetching anime list on page {page}: {all_anime.status_code}")
                total_pages = all_anime.json().get("pagination").get("last_visible_page")
            except Exception as e:
                list_anime_data_not_taken.append(page)
                print(f"Error on page {page}: {e}")
                break

            for anime in all_anime.json().get("data"):
                anime_id = anime.get("mal_id")
                if len(result) >= curr_limit:
                    isDone = True
                    break
                try:
                    #get full anime data for each anime_id
                    last_request_time = check_time_since_last_request(last_request_time)
                    r, last_request_time = get_data(f"anime/{anime_id}/full", metrics, client, anime_id=anime_id)
                    
                    if r.status_code != 200:
                        raise Exception(f"Error fetching full data for anime_id {anime_id}: {r.status_code}")
                    
                    # get anime statistics for each anime_id
                    last_request_time = check_time_since_last_request(last_request_time)
                    r_stats, last_request_time = get_data(f"anime/{anime_id}/statistics", metrics, client, anime_id=anime_id)

                    if r_stats.status_code != 200:
                        raise Exception(f"Error fetching statistics for anime_id {anime_id}: {r_stats.status_code}")

                    #get anime reviews for each anime_id
                    last_request_time = check_time_since_last_request(last_request_time)
                    r_reviews, last_request_time = get_data(f"anime/{anime_id}/reviews", metrics, client, anime_id=anime_id)

                    if r_reviews.status_code != 200:
                        raise Exception(f"Error fetching reviews for anime_id {anime_id}: {r_reviews.status_code}")

                    #get anime episodes for each anime_id
                    last_request_time = check_time_since_last_request(last_request_time)
                    r_episodes, last_request_time = get_data(f"anime/{anime_id}/episodes", metrics, client, anime_id=anime_id)

                    if r_episodes.status_code != 200:  
                        raise Exception(f"Error fetching episodes for anime_id {anime_id}: {r_episodes.status_code}")
                    
                    episode_data = r_episodes.json().get("data")
                    if r_episodes.status_code == 200:
                        total_episodes_pages = r_episodes.json().get("pagination").get("last_visible_page")
                        if total_episodes_pages > 1:
                            for episode_page in range(2, total_episodes_pages + 1):
                                last_request_time = check_time_since_last_request(last_request_time)
                                r_episodes_page, last_request_time = get_data(f"anime/{anime_id}/episodes?page={episode_page}", metrics, client, anime_id=anime_id)
                            
                                if r_episodes_page.status_code != 200:      
                                    raise Exception(f"Error fetching episodes for anime_id {anime_id} on page {episode_page}: {r_episodes_page.status_code}")
                                episode_data.extend(r_episodes_page.json().get("data"))


                    # currently local storage change to s3 later or whatever
                    result.append({
                        "anime_id": anime_id,
                        "anime_data": r.json().get("data"),
                        "anime_statistics": r_stats.json().get("data"),
                        "anime_reviews": r_reviews.json().get("data"),
                        "anime_episodes": episode_data
                    })
                    with open(RESULTS_FILE, "a", encoding="utf-8") as f:
                        f.write(json.dumps(result[-1], ensure_ascii=False) + "\n")
                except Exception as e:
                    anime_data_not_taken.append(anime_id)
                    print(f"Error on anime id {anime_id}: {e}")

            page += 1
            if page > total_pages:   
                isDone = True

            pagination = all_anime.json().get("pagination", {})
            print(pagination)   # look at everything the API gives you
                    
            total_pages = pagination.get("last_visible_page")
            exact_total = pagination.get("items", {}).get("total")   # only if Tensei provides it
                    
            print(f"Last page: {total_pages}")
            print(f"Estimated catalogue size: ~{total_pages * 100}")
            if exact_total:
                print(f"Exact total from API: {exact_total}")

    except Exception as e:
        print(f"An error occurred: {e}")

    finally:
        with open(METRICS_FILE, "w", encoding="utf-8") as f:
            json.dump(metrics, f, ensure_ascii=False, indent=4)
        print(f"Metrics saved to {METRICS_FILE}")

        if client:
            client.close()

    return result, anime_data_not_taken, list_anime_data_not_taken, metrics



## 3. Runs

Four groups of 150 anime from list pages spread across the ID range (pages 1-2, 75-76, 150-151 and 225-226, 100 anime per page), then the 100 most-followed anime. All at 100 requests/min.

In [ ]:
for p in [1, 75, 150, 225]:
    prone_pipeline(curr_limit=150, page=p)

In [ ]:
result, anime_data_not_taken, list_anime_data_not_taken, metrics = prone_pipeline(curr_limit=100, page=1,base_endpoint="anime?limit=100&rating=g%2Cpg%2Cpg13%2Cr17%2Cr&order_by=members&sort=desc", file_name="top_100")

## 4. Analysis

Load every request log and result file into pandas. Each group is one run; `m` has one row per HTTP attempt.

In [35]:
import glob, json, os
from collections import Counter

import numpy as np
import pandas as pd

CATALOGUE_SIZE = None        # fill in from the pagination check, e.g. 29000
TARGET_RPM = 100             # the rate the limiter aims for
RANDOM_PREFIX = "page"       # groups named page1, page75, ... are the random spread


def group_name(path, prefix, suffix):
    return os.path.basename(path)[len(prefix):-len(suffix)]


# metrics: one row per HTTP attempt
frames = []
for path in glob.glob("probe_metrics_*.json"):
    with open(path, encoding="utf-8") as f:
        rows = json.load(f)
    if rows:
        df = pd.DataFrame(rows)
        df["group"] = group_name(path, "probe_metrics_", ".json")
        frames.append(df)
m = pd.concat(frames, ignore_index=True)
m["ok"] = m["status"] == 200
m["status_label"] = m["status"].apply(lambda s: "network_error" if pd.isna(s) else str(int(s)))
ok = m[m["ok"]]

# results: one line per successfully fetched anime
results = {}
for path in glob.glob("probe_results_*.jsonl"):
    with open(path, encoding="utf-8") as f:
        results[group_name(path, "probe_results_", ".jsonl")] = [
            json.loads(line) for line in f if line.strip()
        ]

groups = sorted(m["group"].unique())
random_groups = [g for g in groups if g.startswith(RANDOM_PREFIX)]
print("Groups:", groups)
print("Random groups:", random_groups)
print("Total request rows:", len(m))

Groups: ['page1', 'page150', 'page225', 'page75', 'top_100']
Random groups: ['page1', 'page150', 'page225', 'page75']
Total request rows: 2852


### Status codes

Every response code per group. A 429 or 5xx here would mean we exceeded the limit or the service struggled.

In [ ]:
status_table = pd.crosstab(m["group"], m["status_label"], margins=True)
print(status_table)

errors = m[m["status"].isna()].groupby(["group", "error"]).size()
print("\nNetwork errors by type:\n", errors if len(errors) else "none")

retry_after_values = m.loc[m["status"] == 429, "retry_after"].value_counts(dropna=False)
print("\nRetry-After values on 429s:\n", retry_after_values if len(retry_after_values) else "no 429s")

status_label   200   All
group                   
page1          618   618
page150        602   602
page225        602   602
page75         605   605
top_100        425   425
All           2852  2852

Network errors by type:
 none

Retry-After values on 429s:
 no 429s


### Pacing: did we stay under the limits?

For each request, count how many requests started within the next 1 s and the next 60 s; the maximum is compared with the 4/s and 120/min limits. Start times are used because the server counts requests when they arrive. The second table lists the largest gaps between requests and what came just before them.

In [38]:
def pacing(g):
    t = np.sort(g["started"].to_numpy())
    gaps = np.diff(t)
    duration = t[-1] - t[0]
    idx = np.arange(len(t))
    # for each request, how many requests started in the next 60 s / 1 s (including itself)
    max_60 = int((np.searchsorted(t, t + 60, side="left") - idx).max())
    max_1 = int((np.searchsorted(t, t + 1, side="left") - idx).max())
    return pd.Series({
        "requests": len(t),
        "minutes": duration / 60,
        "avg_per_min": len(t) / (duration / 60) if duration else np.nan,
        "avg_per_sec": len(t) / duration if duration else np.nan,
        "avg_gap_s": gaps.mean(),
        "min_gap_s": gaps.min(),
        "max_gap_s": gaps.max(),
        "max_in_any_60s": max_60,
        "max_in_any_1s": max_1,
    })

pace = pd.DataFrame({g: pacing(d) for g, d in m.groupby("group")}).T
print(pace.round(2))


# the biggest gaps, and what happened just before each one
def largest_gaps(g, n=5):
    g = g.sort_values("started").copy()
    g["gap_before_s"] = g["started"].diff()
    g["prev_endpoint"] = g["endpoint"].shift()
    g["prev_status"] = g["status_label"].shift()
    g["prev_latency_s"] = g["latency_s"].shift()
    return g.nlargest(n, "gap_before_s")[
        ["group", "gap_before_s", "prev_endpoint", "prev_status", "prev_latency_s", "endpoint"]
    ]

print("\nLargest gaps:")
print(pd.concat([largest_gaps(d) for _, d in m.groupby("group")]).round(2))

         requests  minutes  avg_per_min  avg_per_sec  avg_gap_s  min_gap_s  \
page1       618.0     6.17       100.10         1.67        0.6        0.6   
page150     602.0     6.03        99.76         1.66        0.6        0.6   
page225     602.0     6.01       100.10         1.67        0.6        0.6   
page75      605.0     6.04       100.10         1.67        0.6        0.6   
top_100     425.0     4.24       100.17         1.67        0.6        0.6   

         max_gap_s  max_in_any_60s  max_in_any_1s  
page1         0.60           100.0            2.0  
page150       1.81           100.0            2.0  
page225       0.60           100.0            2.0  
page75        0.60           100.0            2.0  
top_100       0.60           100.0            2.0  

Largest gaps:
        group  gap_before_s prev_endpoint prev_status  prev_latency_s  \
565     page1          0.60      episodes         200            0.05   
295     page1          0.60       reviews         200     

### Latency and size per endpoint

Median and p95 latency, and average and maximum response size, per endpoint. Used to decide whether concurrency would help and to estimate the raw layer size.

In [39]:
endpoint_stats = ok.groupby("endpoint").agg(
    calls=("ok", "size"),
    p50_s=("latency_s", "median"),
    p95_s=("latency_s", lambda s: s.quantile(0.95)),
    max_s=("latency_s", "max"),
    avg_kb=("bytes", lambda b: b.mean() / 1024),
    max_kb=("bytes", lambda b: b.max() / 1024),
)
print(endpoint_stats.round(2))

# median latency per endpoint per group (are popular anime slower?)
print(ok.pivot_table(index="endpoint", columns="group", values="latency_s", aggfunc="median").round(2))

            calls  p50_s  p95_s  max_s  avg_kb  max_kb
endpoint                                              
episodes      742   0.05   0.06   0.22   15.78  234.82
full          700   0.05   0.05   0.09    6.11   88.13
list           10   0.15   0.25   0.26  356.55  511.18
reviews       700   0.05   0.08   0.28   38.13  239.77
statistics    700   0.05   0.06   0.09    0.52    0.56
group       page1  page150  page225  page75  top_100
endpoint                                            
episodes     0.05     0.05     0.05    0.05     0.05
full         0.05     0.04     0.05    0.05     0.04
list         0.12     0.15     0.18    0.15     0.19
reviews      0.06     0.05     0.05    0.05     0.07
statistics   0.05     0.05     0.05    0.05     0.05


### Cost per anime

Calls, size and episode pages per anime, by group, plus the most expensive anime. This gives the average cost used for the backfill estimate.

In [40]:
anime_rows = m[m["id"].notna()].copy()
anime_rows["id"] = anime_rows["id"].astype(int)

per_anime = anime_rows.groupby(["group", "id"]).agg(
    calls=("endpoint", "size"),                     # every attempt, since retries cost quota too
    kb=("bytes", lambda b: b.sum() / 1024),
    first=("started", "min"),
    last=("started", "max"),
)
per_anime["seconds"] = per_anime["last"] - per_anime["first"]

episode_pages = (
    ok[(ok["endpoint"] == "episodes") & ok["id"].notna()]
    .assign(id=lambda d: d["id"].astype(int))
    .groupby(["group", "id"]).size()
)
per_anime["episode_pages"] = episode_pages.reindex(per_anime.index).fillna(0).astype(int)

per_anime_summary = per_anime.groupby("group").agg(
    anime=("calls", "size"),
    avg_calls=("calls", "mean"),
    max_calls=("calls", "max"),
    avg_kb=("kb", "mean"),
    max_kb=("kb", "max"),
    avg_seconds=("seconds", "mean"),
    share_multi_episode_pages=("episode_pages", lambda e: (e > 1).mean()),
    max_episode_pages=("episode_pages", "max"),
)
print(per_anime_summary.round(2))

print("\nMost expensive anime:")
print(per_anime.sort_values("calls", ascending=False).head(10)[["calls", "episode_pages", "kb", "seconds"]].round(1))

         anime  avg_calls  max_calls  avg_kb   max_kb  avg_seconds  \
group                                                                
page1      150       4.11         15  104.89  1771.81         1.87   
page150    150       4.00          4   14.35   129.34         1.80   
page225    150       4.00          4   18.37   123.56         1.80   
page75     150       4.02          6   28.28   227.92         1.81   
top_100    100       4.23         15  181.54  1771.81         1.94   

         share_multi_episode_pages  max_episode_pages  
group                                                  
page1                         0.03                 12  
page150                       0.00                  1  
page225                       0.00                  1  
page75                        0.01                  3  
top_100                       0.07                 12  

Most expensive anime:
               calls  episode_pages      kb  seconds
group   id                               

### Cache and rate-limit headers

Whether cached responses (`x-nginx-cache`) are faster, and whether the API sends any rate-limit headers.

In [41]:
cache = (
    ok.assign(cache=ok["nginx_cache"].fillna("none"))
    .groupby("cache")["latency_s"]
    .agg(count="count", median_s="median", p95_s=lambda s: s.quantile(0.95))
)
print(cache.round(3))

ratelimit_headers = {}
for d in m["ratelimit"]:
    if isinstance(d, dict):
        for k, v in d.items():
            ratelimit_headers.setdefault(k, set()).add(v)

if ratelimit_headers:
    for k, vals in ratelimit_headers.items():
        print(f"{k}: {sorted(vals)[:10]}")
else:
    print("No rate-limit headers sent by the API")

       count  median_s  p95_s
cache                        
HIT      714     0.044  0.056
MISS    1944     0.049  0.070
STALE    194     0.045  0.054
No rate-limit headers sent by the API


### Saved vs failed

Anime requested in each run compared with anime saved to the results file.

In [42]:
for g in groups:
    seen = set(anime_rows.loc[anime_rows["group"] == g, "id"])
    saved = {r["anime_id"] for r in results.get(g, [])}
    failed = sorted(seen - saved)
    print(f"{g}: {len(saved)} saved, {len(failed)} failed {failed[:20]}")

page1: 150 saved, 0 failed []
page150: 150 saved, 0 failed []
page225: 150 saved, 0 failed []
page75: 150 saved, 0 failed []
top_100: 100 saved, 0 failed []


### What the data contains

Field names per resource, ratings and types in the sample (to check the rating filter), reviews per anime, how many episodes have a score, and sample image URLs.

In [43]:
all_rows = [r for rows in results.values() for r in rows]

def first_non_empty(key):
    for r in all_rows:
        val = r.get(key)
        if val:
            return val[0] if isinstance(val, list) else val
    return None

# field names per resource
for key in ["anime_data", "anime_statistics", "anime_reviews", "anime_episodes"]:
    sample = first_non_empty(key)
    print(f"\n{key} fields:", sorted(sample.keys()) if isinstance(sample, dict) else sample)

# ratings and types: check no Rx titles slipped through the filter
print("\nRatings:", Counter(r["anime_data"].get("rating") for r in all_rows if r.get("anime_data")))
print("Types:", Counter(r["anime_data"].get("type") for r in all_rows if r.get("anime_data")))

# reviews
review_counts = [len(r.get("anime_reviews") or []) for r in all_rows]
print(f"\nReviews per anime: max {max(review_counts)}, avg {np.mean(review_counts):.1f}, "
      f"no reviews: {sum(c == 0 for c in review_counts)}/{len(review_counts)}")

# episodes: how many have a score (check the field name in the episode fields printed above)
episodes = [e for r in all_rows for e in (r.get("anime_episodes") or [])]
scored = [e for e in episodes if e.get("score") is not None]
print(f"Episodes: {len(episodes)} total, {len(scored)} with a score "
      f"({len(scored) / max(len(episodes), 1):.0%})")

# a few image URLs to test hotlinking in a browser
for r in all_rows[:3]:
    print(r["anime_data"].get("images"))


anime_data fields: ['aired', 'airing', 'approved', 'background', 'broadcast', 'demographics', 'duration', 'episodes', 'explicit_genres', 'external', 'favorites', 'genres', 'images', 'licensors', 'mal_id', 'members', 'moreinfo', 'popularity', 'producers', 'rank', 'rating', 'relations', 'score', 'scored_by', 'season', 'source', 'status', 'streaming', 'studios', 'synopsis', 'theme', 'themes', 'title', 'title_english', 'title_japanese', 'title_synonyms', 'titles', 'trailer', 'type', 'url', 'year']

anime_statistics fields: ['completed', 'dropped', 'on_hold', 'plan_to_watch', 'scores', 'total', 'watching']

anime_reviews fields: ['date', 'episodes_watched', 'is_preliminary', 'is_spoiler', 'mal_id', 'reactions', 'review', 'score', 'tags', 'type', 'url', 'user']

anime_episodes fields: ['aired', 'duration', 'filler', 'forum_url', 'images', 'mal_id', 'recap', 'replies', 'score', 'synopsis', 'title', 'title_japanese', 'title_romanji', 'url']

Ratings: Counter({'PG-13 - Teens 13 or older': 388,

### Summary file

Generates `probe_summary.md` from the tables above. `docs/METRICS.md` is the written-up version.

In [44]:
rand = per_anime.loc[random_groups] if random_groups else per_anime
calls_per_anime = rand["calls"].mean()
kb_per_anime = rand["kb"].mean()
achieved_rpm = pace.loc[random_groups, "avg_per_min"].mean() if random_groups else pace["avg_per_min"].mean()

lines = ["## Tensei probe", ""]
lines += [f"- Groups: {', '.join(groups)} (random spread: {', '.join(random_groups)})",
          f"- Requests made: {len(m)}",
          f"- Calls per anime (random groups): {calls_per_anime:.2f}",
          f"- Raw size per anime (random groups): {kb_per_anime:.1f} KB",
          f"- Achieved rate (random groups): {achieved_rpm:.0f} requests/min"]

if CATALOGUE_SIZE:
    total_calls = CATALOGUE_SIZE * calls_per_anime + CATALOGUE_SIZE / 100   # + list pages
    lines += [f"- Catalogue size: {CATALOGUE_SIZE}",
              f"- Estimated backfill calls: {total_calls:,.0f}",
              f"- Estimated backfill time at achieved rate: {total_calls / achieved_rpm / 60:.1f} h",
              f"- Estimated backfill time at {TARGET_RPM}/min: {total_calls / TARGET_RPM / 60:.1f} h",
              f"- Estimated raw layer size: {CATALOGUE_SIZE * kb_per_anime / 1024 / 1024:.2f} GB"]
else:
    lines.append("- Set CATALOGUE_SIZE to get the backfill estimates")


def md(df):
    try:
        return df.round(2).to_markdown()
    except ImportError:
        return "```\n" + df.round(2).to_string() + "\n```"

lines += ["", "### Status codes", md(status_table),
          "", "### Pacing", md(pace),
          "", "### Latency and size per endpoint", md(endpoint_stats),
          "", "### Cost per anime", md(per_anime_summary),
          "", "### Cache", md(cache),
          "", "### Rate-limit headers",
          ", ".join(ratelimit_headers) if ratelimit_headers else "None sent"]

summary = "\n".join(lines)
with open("probe_summary.md", "w", encoding="utf-8") as f:
    f.write(summary)
print(summary)

## Tensei probe

- Groups: page1, page150, page225, page75, top_100 (random spread: page1, page150, page225, page75)
- Requests made: 2852
- Calls per anime (random groups): 4.03
- Raw size per anime (random groups): 41.5 KB
- Achieved rate (random groups): 100 requests/min
- Set CATALOGUE_SIZE to get the backfill estimates

### Status codes
```
status_label   200   All
group                   
page1          618   618
page150        602   602
page225        602   602
page75         605   605
top_100        425   425
All           2852  2852
```

### Pacing
```
         requests  minutes  avg_per_min  avg_per_sec  avg_gap_s  min_gap_s  max_gap_s  max_in_any_60s  max_in_any_1s
page1       618.0     6.17       100.10         1.67        0.6        0.6       0.60           100.0            2.0
page150     602.0     6.03        99.76         1.66        0.6        0.6       1.81           100.0            2.0
page225     602.0     6.01       100.10         1.67        0.6        0.6       